In [1]:
cell_str = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// Configurazione estrema: Blocchi 64x64, ogni thread gestisce 4x4
#define BM 128
#define BN 128
#define BK 16
#define TM 8
#define TN 8

// ---------------------------------------------------------
// KERNEL CUDA: 2D Register Tiling
// ---------------------------------------------------------
__global__ void sgemm_register_tiled(int n, const float *A, const float *B, float *C) {
    // Memoria Condivisa per i Tile
    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x; // 0..15 (poiché 64/4 = 16)
    int ty = threadIdx.y; // 0..15

    // Indice di partenza per il blocco 4x4 di questo thread
    int row = by * BM + ty * TM;
    int col = bx * BN + tx * TN;

    // I Registri: ultra-veloci, mantengono i risultati parziali del 4x4
    float res[TM][TN] = {0.0f};

    // Indice lineare del thread nel blocco (0..255) per il caricamento collaborativo
    int tid = ty * (BN / TN) + tx;

    // Loop principale sulle fasi K
    for (int p = 0; p < (n + BK - 1) / BK; ++p) {

        // 1. CARICAMENTO COLLABORATIVO (I 256 thread caricano i dati dalla Globale alla Shared)
        // Caricamento del tile di A (64x16 = 1024 elementi -> 4 elementi per thread)
        for (int i = 0; i < (BM * BK) / 256; ++i) {
            int idx = i * 256 + tid;
            int a_row = idx / BK;
            int a_col = idx % BK;
            int g_a_row = by * BM + a_row;
            int g_a_col = p * BK + a_col;
            sA[a_row][a_col] = (g_a_row < n && g_a_col < n) ? A[g_a_row * n + g_a_col] : 0.0f;
        }

        // Caricamento del tile di B (16x64 = 1024 elementi -> 4 elementi per thread)
        for (int i = 0; i < (BK * BN) / 256; ++i) {
            int idx = i * 256 + tid;
            int b_row = idx / BN;
            int b_col = idx % BN;
            int g_b_row = p * BK + b_row;
            int g_b_col = bx * BN + b_col;
            sB[b_row][b_col] = (g_b_row < n && g_b_col < n) ? B[g_b_row * n + g_b_col] : 0.0f;
        }

        __syncthreads();

        // 2. CALCOLO ESTREMO NEI REGISTRI
        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            // Sposta i dati dalla shared ai registri per la moltiplicazione
            float a_reg[TM];
            float b_reg[TN];

            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            // Calcola il blocco 4x4
            for(int i=0; i<TM; i++) {
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }

    // 3. SALVATAGGIO IN MEMORIA GLOBALE
    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row + i;
            int c_col = col + j;
            if (c_row < n && c_col < n) {
                C[c_row * n + c_col] = res[i][j];
            }
        }
    }
}

// ---------------------------------------------------------
// HOST (Passato tutto a FLOAT)
// ---------------------------------------------------------
int main(int argc, char **argv) {
    if (argc < 2) return 1;
    int n = atoi(argv[1]);
    if (n <= 0) return 1;

    size_t bytes = n * n * sizeof(float);

    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // I Thread per blocco sono 16x16 = 256. Ogni thread calcola un 4x4. Totale blocco = 64x64.
    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((n + BN - 1) / BN, (n + BM - 1) / BM);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    sgemm_register_tiled<<<blocksPerGrid, threadsPerBlock>>>(n, d_a, d_b, d_c);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Fast Execution Time: %.4f seconds\n", milliseconds / 1000.0f);

    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    FILE *f = fopen("mat-res-fast.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return 0;
}
'''

with open('my_cuda.cu', 'w') as f:
    f.write(cell_str)

In [2]:
!nvcc -arch=sm_75 -O3 my_cuda.cu -o my_cuda

In [3]:
!nvprof ./my_cuda 17050

==2698== NVPROF is profiling process 2698, command: ./my_cuda 17050
Fast Execution Time: 2.4639 seconds
==2698== Profiling application: ./my_cuda 17050
==2698== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   76.34%  2.46349s         1  2.46349s  2.46349s  2.46349s  sgemm_register_tiled(int, float const *, float const *, float*)
                   16.06%  518.33ms         2  259.17ms  248.52ms  269.81ms  [CUDA memcpy HtoD]
                    7.60%  245.29ms         1  245.29ms  245.29ms  245.29ms  [CUDA memcpy DtoH]
      API calls:   71.08%  2.46350s         1  2.46350s  2.46350s  2.46350s  cudaEventSynchronize
                   22.09%  765.51ms         3  255.17ms  245.68ms  270.04ms  cudaMemcpy
                    6.53%  226.33ms         3  75.442ms  168.32us  225.99ms  cudaMalloc
                    0.16%  5.4587ms         3  1.8196ms  728.83us  2.6323ms  cudaFree
                    0.09%  3.1646ms       114 